# Fuel Economy EDA with sqlframe library

In [1]:
from sqlframe.base.dataframe import BaseDataFrame
from sqlframe.duckdb import DuckDBSession
from sqlframe.duckdb.types import StructType, StructField, StringType, IntegerType
from sqlframe.duckdb import Window
import sqlframe.duckdb.functions as F

# For "prettier" printing of data frame tables in jupyterlab notebook, will convert to polars data frame
import polars as pl
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)

# sqlframe doesn't support a .toPolars() conversion method, so will "monkey patch" one here:
def _to_polars(self) -> pl.DataFrame:
    return pl.from_arrow(self.toArrow())

BaseDataFrame.toPolars = _to_polars

In [2]:
session = DuckDBSession()

In [3]:
# data dictionary can be found at: https://www.fueleconomy.gov/feg/ws/index.shtml#vehicle
df = session.read.csv("https://www.fueleconomy.gov/feg/epadata/vehicles.csv", inferSchema=False)

In [4]:
df.printSchema()

root
 |-- barrels08: double (nullable = true)
 |-- barrelsA08: double (nullable = true)
 |-- charge120: double (nullable = true)
 |-- charge240: double (nullable = true)
 |-- city08: bigint (nullable = true)
 |-- city08U: double (nullable = true)
 |-- cityA08: bigint (nullable = true)
 |-- cityA08U: double (nullable = true)
 |-- cityCD: double (nullable = true)
 |-- cityE: double (nullable = true)
 |-- cityUF: double (nullable = true)
 |-- co2: bigint (nullable = true)
 |-- co2A: bigint (nullable = true)
 |-- co2TailpipeAGpm: double (nullable = true)
 |-- co2TailpipeGpm: double (nullable = true)
 |-- comb08: bigint (nullable = true)
 |-- comb08U: double (nullable = true)
 |-- combA08: bigint (nullable = true)
 |-- combA08U: double (nullable = true)
 |-- combE: double (nullable = true)
 |-- combinedCD: double (nullable = true)
 |-- combinedUF: double (nullable = true)
 |-- cylinders: bigint (nullable = true)
 |-- displ: double (nullable = true)
 |-- drive: string (nullable = true)
 |-- 

#### There are too many columns, so I'll limit to just the ones that I'm interested in:

In [5]:
df_final = (
    df.select(
        F.col("year"),
        F.col("make"),
        F.col("model"),
        F.col("basemodel"),
        F.col("trany").alias("transmission_type"),
        F.col("atvType").alias("alt_fuel_adv_tech_veh_type"),
        F.col("fuelType"),
        F.col("fuelType1"),
        F.col("fuelType2"),
        F.col("charge120").alias("hours_to_fully_charge_120v"),
        F.col("charge240").alias("hours_to_fully_charge_240v"),
        F.col("city08").alias("city_mpg_fuel_type1"),
        F.col("cityA08").alias("city_mpg_fuel_type2"),
        F.col("highway08").alias("hwy_mpg_fuel_type1"),
        F.col("highwayA08").alias("hwy_mpg_fuel_type2"),
        F.col("comb08").alias("comb_mpg_fuel_type1"),
        F.col("combA08").alias("comb_mpg_fuel_type2"),
        F.col("phevCity"),
        F.col("phevHwy"),
        F.col("phevComb"),
    )
)

In [7]:
df_final.toPolars().head()

year,make,model,basemodel,transmission_type,alt_fuel_adv_tech_veh_type,fuelType,fuelType1,fuelType2,hours_to_fully_charge_120v,hours_to_fully_charge_240v,city_mpg_fuel_type1,city_mpg_fuel_type2,hwy_mpg_fuel_type1,hwy_mpg_fuel_type2,comb_mpg_fuel_type1,comb_mpg_fuel_type2,phevCity,phevHwy,phevComb
i64,str,str,str,str,str,str,str,str,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1985,"""Alfa Romeo""","""Spider Veloce 2000""","""Spider""","""Manual 5-spd""",null,"""Regular""","""Regular Gasoline""",null,0.0,0.0,19,0,25,0,21,0,0,0,0
1985,"""Ferrari""","""Testarossa""","""Testarossa""","""Manual 5-spd""",null,"""Regular""","""Regular Gasoline""",null,0.0,0.0,9,0,14,0,11,0,0,0,0
1985,"""Dodge""","""Charger""","""Charger""","""Manual 5-spd""",null,"""Regular""","""Regular Gasoline""",null,0.0,0.0,23,0,33,0,27,0,0,0,0
1985,"""Dodge""","""B150/B250 Wagon 2WD""","""B150/B250 Wagon""","""Automatic 3-spd""",null,"""Regular""","""Regular Gasoline""",null,0.0,0.0,10,0,12,0,11,0,0,0,0
1993,"""Subaru""","""Legacy AWD Turbo""","""Legacy/Outback""","""Manual 5-spd""",null,"""Premium""","""Premium Gasoline""",null,0.0,0.0,17,0,23,0,19,0,0,0,0


## Let's see what distinct values are in the alternate fuel/advance technology vehicle type column:

In [8]:
(
    df_final
    .select(
        F.col("alt_fuel_adv_tech_veh_type")
    )
    .distinct()
).show()

+----------------------------+
| alt_fuel_adv_tech_veh_type |
+----------------------------+
|            None            |
|             EV             |
|           Hybrid           |
|       Plug-in Hybrid       |
|            FFV             |
|           Diesel           |
|            FCV             |
|            eFCV            |
|            CNG             |
|        Bifuel (CNG)        |
|        Bifuel (LPG)        |
+----------------------------+


#### On average, how long does it take for a 2026 model electric vehicle to fully charge using 240 volt charger?

In [9]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%")) &
        (F.col("year") == 2026)
    )
    .agg(
        F.avg(
            F.col("hours_to_fully_charge_240v")
        ).alias("Avg Hours to Fully Charge 240V")
    )
).show()

+--------------------------------+
| Avg Hours to Fully Charge 240V |
+--------------------------------+
|       10.58909657320872        |
+--------------------------------+


#### What is the longest time to charge a 2026 EV vehicle with a 240 volt charger?

In [10]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .agg(
        F.max(
            F.col("hours_to_fully_charge_240v")
        ).alias("Max Hours to Fully Charge 240V")
    )
).show()

+--------------------------------+
| Max Hours to Fully Charge 240V |
+--------------------------------+
|              18.6              |
+--------------------------------+


#### What is the shortest time to charge a 2026 EV vehicle with a 240 volt charger?

In [11]:
(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .agg(
        F.min(
            F.col("hours_to_fully_charge_240v")
        ).alias("Min Hours to Fully Charge 240V")
    )
).show()

+--------------------------------+
| Min Hours to Fully Charge 240V |
+--------------------------------+
|              5.9               |
+--------------------------------+


#### What if I want to know the year, make, and models associated with the longest and shortest charge times?  One way is to use a `Window.partitionBy()`.

Fastest charging models:

In [62]:
w = Window.partitionBy()  # whole dataset

(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .withColumn("min_hours", F.min("hours_to_fully_charge_240v").over(w))
    .filter(F.col("hours_to_fully_charge_240v") == F.col("min_hours"))
    .select(
        "year",
        "make",
        "model",
        "basemodel",
        "fuelType1",
        "fuelType2",
        F.col("hours_to_fully_charge_240v").alias("Min Hours to Fully Charge 240V")
    )
).show()

+------+-----------+----------------------------------+-----------+-------------+-----------+--------------------------------+
| year |    make   |              model               | basemodel |  fuelType1  | fuelType2 | Min Hours to Fully Charge 240V |
+------+-----------+----------------------------------+-----------+-------------+-----------+--------------------------------+
| 2026 |  Cadillac |    OPTIQ RWD (19 kW Charger)     |   OPTIQ   | Electricity |    None   |              5.9               |
| 2026 |  Cadillac |    OPTIQ AWD (19 kW Charger)     |   OPTIQ   | Electricity |    None   |              5.9               |
| 2026 | Chevrolet |  Equinox EV AWD (19 kW Charger)  |  Equinox  | Electricity |    None   |              5.9               |
| 2026 |  Cadillac | OPTIQ AWD V SERIES 19 kW charger |   OPTIQ   | Electricity |    None   |              5.9               |
+------+-----------+----------------------------------+-----------+-------------+-----------+------------------

Slowest charging models:

In [13]:
w = Window.partitionBy()  # whole dataset

(
    df_final
    .filter(
        (F.col("fuelType1").like("%Elec%"))
        & (F.col("year") == 2026)
    )
    .withColumn("max_hours", F.max("hours_to_fully_charge_240v").over(w))
    .filter(F.col("hours_to_fully_charge_240v") == F.col("max_hours"))
    .select(
        "year",
        "make",
        "model",
        F.col("hours_to_fully_charge_240v").alias("Max Hours to Fully Charge 240V")
    )
).show()

+------+-----------+---------------------------------------+--------------------------------+
| year |    make   |                 model                 | Max Hours to Fully Charge 240V |
+------+-----------+---------------------------------------+--------------------------------+
| 2026 | Chevrolet | Silverado EV Ext Range (11kW Charger) |              18.6              |
| 2026 |    GMC    |   SIERRA EV Ext Range (11kW Charger)  |              18.6              |
+------+-----------+---------------------------------------+--------------------------------+


#### With ICE, gasoline vehicles, one would think with each passing model year, the combined MPG should be highest compared to all other previous model years.  But let's confirm by obtaining the average combined MPG for all ICE, gasoline-exclusive vehicles by model year:

In [14]:
# +----------------------------+
# | alt_fuel_adv_tech_veh_type |
# +----------------------------+
# |            None            |
# |             EV             |
# |           Hybrid           |
# |       Plug-in Hybrid       |
# |            FFV             |
# |           Diesel           |
# |            FCV             |
# |            eFCV            |
# |            CNG             |
# |        Bifuel (CNG)        |
# |        Bifuel (LPG)        |
# +----------------------------+


(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .orderBy(F.col("year"))
).toPolars()

year,avg_comb_mpg_fuel_type1
i64,f64
1984,19.121622
1985,19.394686
1986,19.320457
1987,19.164568
1988,19.367607
1989,19.141964
1990,19.031459
1991,18.83806
1992,18.861566


From above we see that recent model years' MPG aren't the highest.

Based on visually looking at the table above, we see that 2018 had highest average combined MPG.  But how do we determine the model year with highest average combined MPG using sqlframe or without having to visually look at the data?

**Using groupBy, then agg():**

In [15]:
(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .orderBy(F.col("avg_comb_mpg_fuel_type1").desc())
    .limit(1)
).toPolars()

year,avg_comb_mpg_fuel_type1
i64,f64
2018,22.825672


**Obtain average, combined MPG, but also add a new column containing the actual max average combined MPG using max(), then Window.partitionBy():**

In [16]:
(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (F.col("fuelType2").isNull() | (F.col("fuelType2") == ""))
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
    )
    .groupBy("year")
    .agg(F.avg("comb_mpg_fuel_type1").alias("avg_comb_mpg_fuel_type1"))
    .withColumn(
        "max_avg_comb_mpg_fuel_type1",
        F.max("avg_comb_mpg_fuel_type1").over(Window.partitionBy())
    )
    .filter(F.col("avg_comb_mpg_fuel_type1") == F.col("max_avg_comb_mpg_fuel_type1"))
).toPolars()

year,avg_comb_mpg_fuel_type1,max_avg_comb_mpg_fuel_type1
i64,f64,f64
2018,22.825672,22.825672


#### Out of 2016 through 2026 models, obtain the top 3 Acura or Honda models with highest average MPG at the basemodel level

Since a basemodel can have 2 or more models, we need to obtain the average for each basemodel.  Then add rank column using dense rank.

Below is an example of why we can have 3 rows/records for 2016 Honda Fit basemodel.  It is because of the different transmissions.

In [55]:
(
    df_final
    .filter(
        (F.col("year") == 2016)
        & (F.col("basemodel") == "Fit")
    )
    .select(
        F.col("year"),
        F.col("make"),
        F.col("model"),
        F.col("transmission_type"),
        F.col("basemodel"),
        F.col("comb_mpg_fuel_type1")
    )
).toPolars().head()

year,make,model,transmission_type,basemodel,comb_mpg_fuel_type1
i64,str,str,str,str,i64
2016,"""Honda""","""Fit""","""Manual 6-spd""","""Fit""",32
2016,"""Honda""","""Fit""","""Automatic (variable gear ratio…","""Fit""",36
2016,"""Honda""","""Fit""","""Automatic (AV-S7)""","""Fit""",34


Additional reason why a basemodel can have multiple models is because of the number of doors:

In [56]:
(
    df_final
    .filter(
        (F.col("year") == 2016)
        & (F.col("basemodel") == "Civic")
    )
    .select(
        F.col("year"),
        F.col("make"),
        F.col("model"),
        F.col("transmission_type"),
        F.col("basemodel"),
        F.col("comb_mpg_fuel_type1")
    )
).toPolars().head()

year,make,model,transmission_type,basemodel,comb_mpg_fuel_type1
i64,str,str,str,str,i64
2016,"""Honda""","""Civic 4Dr""","""Automatic (variable gear ratio…","""Civic""",35
2016,"""Honda""","""Civic 4Dr""","""Manual 6-spd""","""Civic""",31
2016,"""Honda""","""Civic 4Dr""","""Automatic (variable gear ratio…","""Civic""",34
2016,"""Honda""","""Civic 2Dr""","""Automatic (variable gear ratio…","""Civic""",35
2016,"""Honda""","""Civic 2Dr""","""Manual 6-spd""","""Civic""",30


So to eliminate the multiple rows or collapse them, need to aggregate and get the average of the combined MPG:

In [59]:
(
    df_final
    .filter(
        (F.col("year") == 2016)
        & (F.col("basemodel") == "Civic")
    )
    .select(
        F.col("year"),
        F.col("make"),
        F.col("model"),
        F.col("transmission_type"),
        F.col("basemodel"),
        F.col("comb_mpg_fuel_type1")
    )
    .groupBy(
        F.col("year"),
        F.col("make"),
        F.col("basemodel")
    )
    .agg(
        F.avg(F.col("comb_mpg_fuel_type1")).alias("avg_comb_mpg")
    )
).toPolars().head()

year,make,basemodel,avg_comb_mpg
i64,str,str,f64
2016,"""Honda""","""Civic""",33.166667


Now, back to our goal: Out of 2016 through 2026 models, obtain the top 3 Acura or Honda models with highest average MPG at the basemodel level

We can use combination of groupyBy/agg and then dense rank with a window partitioned by year.

In [49]:
w = Window.partitionBy("year").orderBy(F.col("avg_comb_mpg").desc())

(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
        & (
            F.col("year").between(2016,2026)
        )
        & (
            F.col("make").isin("Honda","Acura")
        )
    )
    .groupBy(
        "year","basemodel"
    )
    .agg(
        F.avg(F.col("comb_mpg_fuel_type1")).alias("avg_comb_mpg")
    )
    .withColumn(
        "rank", F.dense_rank().over(w)
    )
    # # Comment out .filter to see the unfiltered version
    .filter(
        F.col("rank").between(1,3)
    )
    .orderBy(
        F.col("year")
        , F.col("rank")
    )
).toPolars()

year,basemodel,avg_comb_mpg,rank
i64,str,f64,i64
2016,"""Fit""",34.0,1
2016,"""Civic""",33.166667,2
2016,"""HR-V""",29.6,3
2017,"""Fit""",34.0,1
2017,"""Civic""",32.933333,2
2017,"""HR-V""",29.6,3
2018,"""Fit""",33.333333,1
2018,"""Civic""",32.933333,2
2018,"""HR-V""",29.6,3


From above, we get expected results.  The smaller cars are ranked high for fuel efficiency: Fit, Civic, HR-V.

#### Now, let's get the top 3 least fuel efficient Acura or Honda models from 2016 through 2026 model year at the base model level:

In [61]:
w = Window.partitionBy("year").orderBy(F.col("avg_comb_mpg").asc())

(
    df_final
    .filter(
        F.col("fuelType1").like("%Gasoline%")
        & (
            F.col("alt_fuel_adv_tech_veh_type").isNull()
            | ~F.col("alt_fuel_adv_tech_veh_type").isin("EV", "Hybrid", "Plug-in Hybrid", "FFV", "Diesel","FCV","eFCV","CNG","Bifuel (CNG)","Bifuel (LPG)")
        )
        & (
            F.col("year").between(2016,2026)
        )
        & (
            F.col("make").isin("Honda","Acura")
        )
    )
    .groupBy(
        "year","basemodel"
    )
    .agg(
        F.avg(F.col("comb_mpg_fuel_type1")).alias("avg_comb_mpg")
    )
    .withColumn(
        "rank", F.dense_rank().over(w)
    )
    # # Comment out .filter to see the unfiltered version
    .filter(
        F.col("rank").between(1,3)
    )
    .orderBy(
        F.col("year")
        , F.col("rank")
    )
).toPolars()

year,basemodel,avg_comb_mpg,rank
i64,str,f64,i64
2016,"""Odyssey""",22.0,1
2016,"""MDX""",22.0,1
2016,"""Pilot""",22.0,1
2016,"""RDX""",22.5,2
2016,"""RLX""",23.0,3
2017,"""Ridgeline""",21.5,1
2017,"""Pilot""",22.0,2
2017,"""MDX""",22.0,2
2017,"""Odyssey""",22.0,2


No surprise that the larger vehicles are the worst performers.